# `lg_workflow_agent` — Simple Checks & Testing

This notebook performs incremental sanity checks on the workflow agent at `src/lg_workflow_agent`:

1. Environment & imports
2. Tool checks (`fetch_trends`, URL validator)
3. Prompt / state shape checks
4. Classifier node in isolation
5. Task generator node in isolation
6. Aggregator node with stubbed sub-agent outputs
7. Validator node with broken-link rewrite path (no LLM)
8. Full end-to-end run (sync)
9. Streaming run with per-step trace

Run cells top-to-bottom. Cells 4-6 and 8-9 require `GOOGLE_API_KEY`.

In [11]:
# 1. Environment & imports
import os
import sys
from pathlib import Path

# Make `src` importable when running the notebook from tests/
BACKEND_ROOT = Path.cwd()
if BACKEND_ROOT.name == "tests":
    BACKEND_ROOT = BACKEND_ROOT.parent
if str(BACKEND_ROOT) not in sys.path:
    sys.path.insert(0, str(BACKEND_ROOT))

from dotenv import load_dotenv
load_dotenv(BACKEND_ROOT / ".env")

print("BACKEND_ROOT:", BACKEND_ROOT)
print("GOOGLE_API_KEY set:", bool(os.environ.get("GOOGLE_API_KEY")))
print("DEEP_AGENT_MODEL:", os.environ.get("DEEP_AGENT_MODEL", "gemini-2.5-flash (default)"))
print("QDRANT_URL set:", bool(os.environ.get("QDRANT_URL")))

BACKEND_ROOT: /Users/kumarnishant/aion-ai-research/backend
GOOGLE_API_KEY set: True
DEEP_AGENT_MODEL: google_genai:gemini-2.5-flash
QDRANT_URL set: True


## 2. Tool checks

`fetch_trends` (HTTP) and the URL validator (`validate_url`) used by the Validator node.

In [12]:
from src.lg_workflow_agent.tools import (
    fetch_trends,
    validate_url,
    validate_urls,
    extract_urls,
)

# fetch_trends is a LangChain @tool — invoke via .invoke({...})
sample = fetch_trends.invoke({"source": "hackernews", "topic": "langgraph", "limit": 3, "period": "week"})
print("fetch_trends preview:", (sample or "")[:300], "...\n")

# URL helpers
text = "See https://www.python.org and https://this-domain-should-not-exist-xyz123.invalid for refs."
urls = extract_urls(text)
print("extracted:", urls)
print("validation:", validate_urls(urls))

fetch_trends preview: {"results":[{"title":"Show HN: Graph-flow – LangGraph-inspired AI agent workflows in Rust","url":"https://github.com/a-agmon/rs-graph-llm","source":"hackernews","metadata":{"points":2,"num_comments":0,"engagement_ratio":0.0,"discussion_url":"https://news.ycombinator.com/item?id=47925524","author":"a ...

extracted: ['https://this-domain-should-not-exist-xyz123.invalid', 'https://www.python.org']
validation: {'https://this-domain-should-not-exist-xyz123.invalid': False, 'https://www.python.org': True}


## 3. Prompt / state shape checks

Quick sanity that prompts format correctly and the state TypedDict carries expected keys.

In [13]:
from src.lg_workflow_agent import prompts as P
from src.lg_workflow_agent.state import WorkflowState
from src.lg_workflow_agent.nodes import ROLES_BY_TYPE, ROLE_PROMPTS

# Required state keys
required_keys = {
    "query", "task_id", "query_type", "subtasks", "worker_payloads",
    "worker_outputs", "aggregated", "draft_report", "final_report",
    "validation_feedback", "invalid_references", "rewrite_iterations",
}
missing = required_keys - set(WorkflowState.__annotations__.keys())
assert not missing, f"State missing keys: {missing}"
print("State keys OK")

# Roles per type
for qt in ["blog", "comparative", "deep_research", "summary"]:
    roles = ROLES_BY_TYPE[qt]
    for r in roles:
        assert r in ROLE_PROMPTS, f"missing prompt for role {r}"
    print(f"  {qt:14s} -> {roles}")

# Prompt formatting smoke
print(P.CLASSIFIER_PROMPT.format(query="hello")[:120])
print(P.TASK_GENERATOR_PROMPT.format(query="x", query_type="blog", roles="- web_research")[:120])

State keys OK
  blog           -> ['web_research', 'content_drafting']
  comparative    -> ['web_research', 'content_drafting']
  deep_research  -> ['data_collection', 'statistics', 'citation']
  summary        -> ['web_research', 'content_drafting']
You are the Query Classifier in a research content workflow.

Classify the user's query into EXACTLY ONE of these catego
You are the Task Generation node.
The query has been classified as: blog.

Available specialized sub-agent roles for que


## 4. Classifier node (isolated)

Initialize an LLM and run only the classifier on a few sample queries.

In [14]:
from langchain_google_genai import ChatGoogleGenerativeAI
from src.lg_workflow_agent.nodes import (
    create_node_classifier,
    create_node_task_generator,
    create_node_aggregator,
    create_node_validator,
)

model_name = os.environ.get("DEEP_AGENT_MODEL", "gemini-2.5-flash")
if model_name.startswith("google_genai:"):
    model_name = model_name.split(":", 1)[1]

llm = ChatGoogleGenerativeAI(model=model_name, temperature=0.0)

# db=None: skip persistence to focus on logic
classify = create_node_classifier(llm, db=None)

samples = [
    "Compare LangGraph vs CrewAI vs AutoGen for multi-agent orchestration",
    "Write a blog post about async Python in 2026",
    "Give me a quick summary of vector databases",
    "Conduct a deep research investigation on quantization-aware training benchmarks",
]
for q in samples:
    out = classify({"query": q, "task_id": "demo"})
    print(f"{q[:55]:55s} -> {out['query_type']:14s} | {out['classification_rationale']}")

Compare LangGraph vs CrewAI vs AutoGen for multi-agent  -> comparative    | The query explicitly asks to 'Compare' three specific tools (LangGraph, CrewAI, AutoGen) against each other for a particular use case.
Write a blog post about async Python in 2026            -> blog           | The user explicitly requested a 'blog post' which falls under the 'blog' category for informal and explanatory articles.
Give me a quick summary of vector databases             -> summary        | The user explicitly asks for a 'quick summary' of vector databases, indicating a need for a short factual digest.
Conduct a deep research investigation on quantization-a -> deep_research  | The user explicitly requests a 'deep research investigation' on a technical topic, indicating a need for rigorous, detailed analysis and potentially benchmarks.


## 5. Task generator node (isolated)

Verify the task generator decomposes a query and assigns roles allowed by the query type.

In [5]:
task_gen = create_node_task_generator(llm, db=None)

state = {
    "query": "Compare LangGraph vs CrewAI for multi-agent orchestration",
    "query_type": "comparative",
    "task_id": "demo",
}
result = task_gen(state)

print(f"Generated {len(result['subtasks'])} subtasks:")
for st in result["subtasks"]:
    print(f"  [{st['id']}] role={st['role']:18s} task={st['task']}")
print("\nWorker payloads (first):")
print(result["worker_payloads"][0])

# Assert role constraints
allowed = set(ROLES_BY_TYPE["comparative"])
assert all(st["role"] in allowed for st in result["subtasks"]), "Role outside allowed set"
print("\nRole assignment OK")

Generated 3 subtasks:
  [s1] role=web_research       task=Research LangGraph's core features, architecture, benefits, and limitations specifically for multi-agent orchestration.
  [s2] role=web_research       task=Research CrewAI's core features, architecture, benefits, and limitations specifically for multi-agent orchestration.
  [s3] role=content_drafting   task=Draft a comparative analysis of LangGraph vs. CrewAI for multi-agent orchestration, highlighting their key differences, strengths, weaknesses, and ideal use cases based on the research findings.

Worker payloads (first):
{'task_id': 'demo', 'query': 'Compare LangGraph vs CrewAI for multi-agent orchestration', 'subtask_id': 's1', 'role': 'web_research', 'task': "Research LangGraph's core features, architecture, benefits, and limitations specifically for multi-agent orchestration."}

Role assignment OK


## 6. Aggregator with stubbed sub-agent outputs

Skip real sub-agent calls and feed pre-baked outputs to confirm the aggregator
returns a structured `{metadata, sections, references}` object.

In [6]:
import json

aggregate = create_node_aggregator(llm, db=None)

stub_outputs = [
    {
        "subtask_id": "s1",
        "role": "web_research",
        "task": "Research LangGraph orchestration model",
        "output": (
            "## Findings\n"
            "LangGraph models workflows as a stateful graph with explicit nodes and edges [1].\n"
            "It supports streaming and human-in-the-loop checkpoints [2].\n"
            "## Sources\n"
            "[1] LangGraph Docs - https://langchain-ai.github.io/langgraph/\n"
            "[2] LangGraph blog - https://blog.langchain.dev/langgraph/"
        ),
    },
    {
        "subtask_id": "s2",
        "role": "content_drafting",
        "task": "Research CrewAI orchestration model",
        "output": (
            "## Draft\n"
            "CrewAI uses a role-based crew abstraction with sequential or hierarchical processes [1].\n"
            "## Sources\n"
            "[1] CrewAI Docs - https://docs.crewai.com/"
        ),
    },
]

state = {
    "query": "Compare LangGraph vs CrewAI",
    "query_type": "comparative",
    "task_id": "demo",
    "worker_outputs": stub_outputs,
}
agg = aggregate(state)["aggregated"]
print(json.dumps(agg, indent=2)[:1500])

{
  "metadata": {
    "query": "Compare LangGraph vs CrewAI",
    "query_type": "comparative",
    "num_sources": 3
  },
  "sections": [
    {
      "title": "LangGraph Overview",
      "content": "LangGraph models workflows as a stateful graph with explicit nodes and edges [1]. It also supports streaming and human-in-the-loop checkpoints [2]."
    },
    {
      "title": "CrewAI Overview",
      "content": "CrewAI utilizes a role-based crew abstraction, allowing for sequential or hierarchical processes [3]."
    }
  ],
  "references": [
    {
      "id": 1,
      "title": "LangGraph Docs",
      "url": "https://langchain-ai.github.io/langgraph/"
    },
    {
      "id": 2,
      "title": "LangGraph blog",
      "url": "https://blog.langchain.dev/langgraph/"
    },
    {
      "id": 3,
      "title": "CrewAI Docs",
      "url": "https://docs.crewai.com/"
    }
  ]
}


## 7. Validator (no LLM)

The validator extracts URLs from the draft, verifies them, and signals a rewrite when broken refs are found.

In [7]:
validator = create_node_validator(db=None)

draft_good = "Read more at https://www.python.org for details."
draft_bad = (
    "Reference [1] points to https://www.python.org but [2] is broken: "
    "https://this-domain-should-not-exist-xyz123.invalid"
)

print("good draft:")
print(validator({"draft_report": draft_good, "rewrite_iterations": 0}))

print("\nbad draft (1st pass -> rewrite):")
print(validator({"draft_report": draft_bad, "rewrite_iterations": 0}))

print("\nbad draft (after MAX_REWRITES -> forced finish):")
print(validator({"draft_report": draft_bad, "rewrite_iterations": 2}))

good draft:
{'final_report': 'Read more at https://www.python.org for details.', 'validation_feedback': 'VALID', 'invalid_references': []}

bad draft (1st pass -> rewrite):
{'validation_feedback': 'BROKEN_REFS: 1 link(s) failed', 'invalid_references': ['https://this-domain-should-not-exist-xyz123.invalid'], 'rewrite_iterations': 1}

bad draft (after MAX_REWRITES -> forced finish):
{'final_report': 'Reference [1] points to https://www.python.org but [2] is broken: [broken link removed]', 'validation_feedback': 'FORCED_FINISH after 2 rewrites', 'invalid_references': []}


## 8. End-to-end (sync)

Build a fresh `WorkflowAgent` and run a single query through the full graph.

In [8]:
from src.lg_workflow_agent import WorkflowAgent

agent = WorkflowAgent()
agent.build()
print("Agent ready:", agent.is_ready)

QUERY = "Give a short summary of LangGraph for multi-agent orchestration"
report = agent.invoke(QUERY)
print("\n--- FINAL REPORT ---\n")
print(report)

Agent ready: True
Cleaned up intermediate data for task: e950c45a-2b5c-490b-ae2b-d3bda53e249b

--- FINAL REPORT ---

# Short Summary of LangGraph for Multi-Agent Orchestration

## Introduction to LangGraph
LangGraph is a Python library built on top of LangChain, specifically designed for constructing "resilient language agents as graphs" [5]. Its fundamental concept involves representing agentic applications as stateful, cyclic graphs, which allows for more complex and robust agent behaviors compared to traditional sequential chains [5]. This graph-based approach enables developers to define various nodes—representing agents, tools, or functions—and edges, which dictate the flow of execution within an application. The application's state is passed between these nodes, facilitating iterative processing and dynamic decision-making [5].

## LangGraph for Multi-Agent Orchestration
LangGraph significantly enhances multi-agent orchestration by providing a structured and explicit method to ma

## 9. Streaming run with per-step trace

Use `astream` to observe how state evolves across nodes (classifier, task_generator, sub-agents, aggregator, writer, validator, cleanup).

In [16]:
import json

STREAM_QUERY = "Conduct a deep research investigation on quantization-aware training benchmarks"

trace = []
final_report = ""
event_count = 0
async for event in agent.astream(STREAM_QUERY):
    step = event.get("step")
    data = event.get("data", {})
    print(f"-----------------------\nEvent: step={step} |\n  data : ={json.dumps(data, indent=2)} \n\n~~~~~~~~~~~~~~~~~~~~~~~~~~")
    # keys = sorted(k for k in data.keys() if k != "messages")
    # line = f"[{step}] keys={keys}"
    # if data.get("query_type"):
    #     line += f" | query_type={data['query_type']}"
    # if data.get("subtasks"):
    #     line += f" | subtasks={len(data['subtasks'])}"
    # if data.get("validation_feedback"):
    #     line += f" | validation={data['validation_feedback']}"
    # if data.get("web_research_agent"):
    #     line += f" | web_research_agent={data['web_research_agent']}"
    # print(line)
    # trace.append(line)
    if data.get("final_report"):
        final_report = data["final_report"]
    elif data.get("draft_report"):
        final_report = data["draft_report"]
    event_count += 1

print(f"\nTotal events: {event_count}")
print("\n--- FINAL REPORT ---\n")
print(final_report or "(no report)")

-----------------------
Event: step=step: classifier |
  data : ={
  "query_type": "deep_research",
  "classification_rationale": "The user explicitly requests a 'deep research investigation' on a technical topic, indicating a need for rigorous, detailed, and potentially citation-heavy content."
} 

~~~~~~~~~~~~~~~~~~~~~~~~~~
-----------------------
Event: step=step: task_generator |
  data : ={
  "subtasks": [
    {
      "id": "s1",
      "role": "data_collection",
      "task": "Collect comprehensive definitions and explanations of Quantization-Aware Training (QAT), including its core principles, common techniques, and the motivations behind its use. Additionally, identify the typical components and general methodologies employed when benchmarking QAT models, such as standard datasets, model architectures, and evaluation metrics.",
      "status": "pending"
    },
    {
      "id": "s2",
      "role": "data_collection",
      "task": "Identify and detail specific, widely recognized 

In [18]:
import json

STREAM_QUERY = "what is latest work in data science"

trace = []
final_report = ""
event_count = 0
async for event in agent.astream(STREAM_QUERY):
    step = event.get("step")
    data = event.get("data", {})
    print(f"-----------------------\nEvent: step={step} |\n  data : ={json.dumps(data, indent=2)} \n\n~~~~~~~~~~~~~~~~~~~~~~~~~~")
    # keys = sorted(k for k in data.keys() if k != "messages")
    # line = f"[{step}] keys={keys}"
    # if data.get("query_type"):
    #     line += f" | query_type={data['query_type']}"
    # if data.get("subtasks"):
    #     line += f" | subtasks={len(data['subtasks'])}"
    # if data.get("validation_feedback"):
    #     line += f" | validation={data['validation_feedback']}"
    # if data.get("web_research_agent"):
    #     line += f" | web_research_agent={data['web_research_agent']}"
    # print(line)
    # trace.append(line)
    if data.get("final_report"):
        final_report = data["final_report"]
    elif data.get("draft_report"):
        final_report = data["draft_report"]
    event_count += 1

print(f"\nTotal events: {event_count}")
print("\n--- FINAL REPORT ---\n")
print(final_report or "(no report)")

-----------------------
Event: step=step: classifier |
  data : ={
  "query_type": "summary",
  "classification_rationale": "The user is asking for an overview of recent developments in a broad field, which aligns with a short factual digest or overview."
} 

~~~~~~~~~~~~~~~~~~~~~~~~~~
-----------------------
Event: step=step: task_generator |
  data : ={
  "subtasks": [
    {
      "id": "s1",
      "role": "web_research",
      "task": "Identify and gather information on the most recent trends, significant breakthroughs, and notable research/applications in data science from the last 1-2 years. Focus on areas like new algorithms, tools, methodologies, and impactful use cases.",
      "status": "pending"
    },
    {
      "id": "s2",
      "role": "content_drafting",
      "task": "Synthesize the gathered research findings into a concise summary outlining the latest work and key developments in data science.",
      "status": "pending"
    }
  ],
  "worker_payloads": [
    {
      "t